# 065 — Advanced Diagnosis

## Oracle ranker: how much of the shortfall is selection loss vs. never-proposed?

Automates the manual `manual_inspection/discards_valid/` analysis from `063-candidate-analysis.ipynb`
using the taxonomy (Appendix C.4). `INST` spans mark every candidate distractor the model mentioned;
for each trace we extract those candidates, union them with the final selected distractors into a
**proposed-candidate pool**, and compute the **oracle proportional match** over that pool (same LLM
judge as the paper, per-GT coverage). `oracle_pm - actual_pm` is the share of human distractors
proposed but dropped -> Table 26, Section 4.3.

Covers 8 cells: {DeepSeek, GLM} × {reasoning, CoT} × {Eedi, SciQ}, on the annotated subset.

In [ ]:
import os, ast, json, glob, pickle, re
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import scipy.stats as st
from openai import OpenAI
from dotenv import load_dotenv

from src.datasets import get_or_create_dataset
from src.equality import MathSemanticEqualityChecker, ScienceSemanticEqualityChecker
from src.model_configurations import gpt_4_1_mini_det_config
from src.prompt_util import prompt_openai

load_dotenv()

STEMS = {
    ("deepseek", "reasoning"): "deepseek-naive-deepseek-reasoner",
    ("deepseek", "cot"):       "deepseek-naive-cot-deepseek-chat",
    ("glm", "reasoning"):      "openrouter-naive-z-ai_glm-4.7-reasoner",
    ("glm", "cot"):            "openrouter-naive-cot-z-ai_glm-4.7-chat",
}
DATAFOLDER = {"eedi": "eedi_data", "sciq": "sciq_data"}
BUCKETS = ("high_match_solvable", "low_match_solvable")
CELLS = [(d, r, m) for d in ("eedi", "sciq") for r in ("reasoning", "cot") for m in ("deepseek", "glm")]

eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
sciq_dataset = get_or_create_dataset("sciq_data", n_limit=500)
datasets_by_datafolder = {"eedi_data": eedi_dataset, "sciq_data": sciq_dataset}
print(f"eedi: {len(eedi_dataset)} | sciq: {len(sciq_dataset)}")

In [ ]:
# Equality judge = identical to the paper (gpt-4.1-mini, deterministic); reuse the pickled caches
# so overlapping comparisons with the main performance runs are free.
equality_model_config = gpt_4_1_mini_det_config
client = OpenAI(base_url=equality_model_config["base_url"],
                api_key=os.environ.get(equality_model_config["api_key_var"]))

MATH_CHECK_PATH = "cache/math_semantic_equivalence_checker.pkl"
SCI_CHECK_PATH  = "cache/science_semantic_equivalence_checker.pkl"

math_checker = (MathSemanticEqualityChecker.load(client, MATH_CHECK_PATH)
                if os.path.exists(MATH_CHECK_PATH)
                else MathSemanticEqualityChecker(client, equality_model_config))
sci_checker  = (ScienceSemanticEqualityChecker.load(client, SCI_CHECK_PATH)
                if os.path.exists(SCI_CHECK_PATH)
                else ScienceSemanticEqualityChecker(client, equality_model_config))
checker_by_dataset = {"eedi": math_checker, "sciq": sci_checker}
print("math memo:", len(math_checker.memoization), "| sci memo:", len(sci_checker.memoization))

In [ ]:
# ---------- INST window reconstruction (from the `annotation` column) + cached candidate extraction ----------
# NOTE: annotation_sequence offsets index into the `annotation` column (tags inline), NOT the clean
# `trace` -- 871/885 rows have offsets past len(trace). So we parse the `annotation` column directly:
# each <INST> sits immediately AFTER the candidate value it marks. We take the ~220 chars before each
# <INST>, strip the other inline tags, and re-append <INST> so the excerpt ends exactly at the candidate.
TAG_LABELS = ["INTER","LINK","CORR","ERR_DESC","ERR_SIM","RECON","PLAUS","DISCR","CURATE","INST"]
_TAGRE = re.compile("<(" + "|".join(TAG_LABELS) + ")>")

def inst_windows(annotation, before=220):
    """One window per <INST> in the annotation column: the preceding ~`before` chars with inline
    tags stripped, ending exactly at the candidate value that <INST> marks."""
    if not isinstance(annotation, str):
        return []
    out = []
    for m in re.finditer(r"<INST>", annotation):
        pre = _TAGRE.sub("", annotation[:m.start()])[-before:].strip()
        if pre:
            out.append(pre)
    return out

EXTRACT_CACHE_PATH = "cache/inst_candidate_extraction_v3mech.pkl"
_extract_cache = pickle.load(open(EXTRACT_CACHE_PATH, "rb")) if os.path.exists(EXTRACT_CACHE_PATH) else {}

# Mechanical, purely-positional extraction prompt (v3): take the value immediately before <INST>,
# not any earlier candidate. Validated on 100 held-out windows at ~2% error.
EXTRACT_SYS = (
    "You are given an excerpt from an LLM's reasoning while it invents wrong answer options "
    "(distractors) for a multiple-choice question. The excerpt ends with a <INST> marker. Immediately to "
    "the LEFT of that marker is one candidate distractor value that the reasoning just named. Your job is "
    "purely positional: output the value token(s) directly before <INST> -- not any other value appearing "
    "earlier in the excerpt. Clean it up: strip surrounding markdown/formatting, quotes, enumeration or "
    "field labels (e.g. 'Distractor1:', 'Idea 3:', '3.', 'F ='), and any trailing bracketed or "
    "parenthetical comment. Use the question and correct answer only to judge where the value begins. "
    "Output ONLY that value, or exactly NONE if there is no concrete answer value immediately before the marker."
)

def _extract_one(problem, correct, window):
    key = (problem, window)
    if key in _extract_cache:
        return _extract_cache[key]
    user = (f"<question>{problem}</question>\n<correct_answer>{correct}</correct_answer>\n"
            f"<excerpt>{window}<INST></excerpt>")
    try:
        out = prompt_openai(client, EXTRACT_SYS, user, equality_model_config)
    except Exception as e:
        print("extract error:", e)
        return None
    v = None if out.strip().upper() == "NONE" else out.strip()
    _extract_cache[key] = v
    return v

def extract_candidates(problem, correct, windows, max_workers=8):
    """One value per window; dedupe (case-insensitive, order-preserving); drop NONE."""
    todo = [w for w in windows if (problem, w) not in _extract_cache]
    if todo:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            list(ex.map(lambda w: _extract_one(problem, correct, w), todo))
    seen, out = set(), []
    for w in windows:
        v = _extract_cache.get((problem, w))
        if v and v.lower() not in seen:
            seen.add(v.lower())
            out.append(v)
    return out

# quick smoke test on one trace
_df = pd.read_csv("eedi_data/joint_results/annotated/deepseek-naive-deepseek-reasoner_low_match_solvable_annot_parsed.csv")
_r = _df.iloc[0]; _dp = eedi_dataset[int(_r["Id"])]
_w = inst_windows(_r["annotation"])
print(f"Id {_r['Id']}: {len(_w)} INST windows -> candidates:",
      extract_candidates(_dp["Problem"]["Question"], _dp["Choices"]["CorrectAnswer"], _w))

In [ ]:
# ---------- oracle metric + population (IPW) helpers ----------
def oracle_proportional_match(checker, problem, candidates, groundtruths):
    """Per-GT coverage in [0,1]: fraction of ground-truth distractors matched by ANY candidate.
    (NOT src.evaluation.proportional_match, which sums matching pairs and is UNCAPPED / can
    exceed 1; we cap so oracle and actual are on the same footing and oracle >= actual holds.)"""
    if not groundtruths:
        return float("nan")
    return sum(any(checker.is_equal(problem, c, g) for c in candidates) for g in groundtruths) / len(groundtruths)

def population_bucket_sizes(data_folder, run):
    """Population counts of solvable traces with pm>0.5 / pm<0.5 (the sampler's frame). From 064."""
    with open(f"{data_folder}/joint_results/{run}_responses_by_datapointid.json") as f:
        responses = json.load(f)
    res = pd.read_csv(f"{data_folder}/joint_results/{run}_results.csv").set_index("Id")
    ds = datasets_by_datafolder[data_folder]
    n_high = n_low = 0
    for k in responses:
        kid = int(k)
        if kid not in res.index or not ds[kid]["Problem"]["Solvable"]:
            continue
        pm = res.loc[kid, "proportional_match"]
        if pm > 0.5:   n_high += 1
        elif pm < 0.5: n_low  += 1
    return n_high, n_low

def gt_distractors(dataset, dpid):
    return dataset[int(dpid)]["Choices"]["Distractors"]

# --- SELECTED (final) distractors: the clean distractorN_answer fields from the responses JSON,
# i.e. exactly what the paper's proportional_match was computed from. We use these (not the
# annotated CSV 'distractors' column, whose parse can fail) as the "selected set". ---
_responses_cache = {}
def load_responses(data_folder, run):
    key = (data_folder, run)
    if key not in _responses_cache:
        with open(f"{data_folder}/joint_results/{run}_responses_by_datapointid.json") as f:
            _responses_cache[key] = json.load(f)
    return _responses_cache[key]

def selected_distractors(responses, dpid):
    o = responses.get(str(dpid), {})
    return [o[f"distractor{i}_answer"] for i in (1, 2, 3)
            if isinstance(o.get(f"distractor{i}_answer"), str) and o[f"distractor{i}_answer"].strip()]


In [ ]:
# ---------- per-trace oracle PM across the 8 cells (judge sweep; caches persisted per cell) ----------
# actual_pm and oracle_pm both use the SAME capped per-GT coverage metric, so oracle >= actual by
# construction (the oracle pool is selected-set UNION proposed candidates). csv_pm keeps the
# original (uncapped, pair-summing) value for reference only. Candidates come from INST windows
# reconstructed from the `annotation` column (see above), extracted with the mechanical v3 prompt.
rows = []
for dataset_key, regime, model in CELLS:
    data_folder = DATAFOLDER[dataset_key]
    ds = datasets_by_datafolder[data_folder]
    checker = checker_by_dataset[dataset_key]
    run = STEMS[(model, regime)]
    responses = load_responses(data_folder, run)
    n_high, n_low = population_bucket_sizes(data_folder, run)
    for bucket, n_pop in (("high_match_solvable", n_high), ("low_match_solvable", n_low)):
        path = f"{data_folder}/joint_results/annotated/{run}_{bucket}_annot_parsed.csv"
        assert "long_problems" not in path
        if not os.path.exists(path):
            print("missing:", path)
            continue
        df = pd.read_csv(path)
        w = n_pop / len(df) if len(df) else 0.0   # inverse sampling probability
        for _, r in df.iterrows():
            dpid = int(r["Id"])
            problem = ds[dpid]["Problem"]["Question"]
            correct = ds[dpid]["Choices"]["CorrectAnswer"]
            gts = gt_distractors(ds, dpid)
            selected = selected_distractors(responses, dpid)          # final 3 (clean)
            windows = inst_windows(r.get("annotation", ""))           # INST windows from annotation col
            cand = extract_candidates(problem, correct, windows)      # proposed in reasoning
            pool = selected + cand                                    # selected OR proposed
            rows.append({
                "dataset": dataset_key, "regime": regime, "model": model, "run": run,
                "bucket": bucket, "Id": dpid, "weight": w,
                "actual_pm": oracle_proportional_match(checker, problem, selected, gts),
                "oracle_pm": oracle_proportional_match(checker, problem, pool, gts),
                "csv_pm": float(r["proportional_match"]),
                "n_gt": len(gts), "n_sel": len(selected), "n_inst": len(windows), "n_cand": len(cand),
            })
    # persist caches incrementally so a rerun makes ~no new API calls
    pickle.dump(_extract_cache, open(EXTRACT_CACHE_PATH, "wb"))
    math_checker.save(MATH_CHECK_PATH)
    sci_checker.save(SCI_CHECK_PATH)
    print(f"done: {dataset_key}/{regime}/{model}")

per_trace = pd.DataFrame(rows).drop_duplicates(subset=["run", "Id"]).reset_index(drop=True)
per_trace.to_csv("cache/oracle_per_trace.csv", index=False)
print(f"\n{len(per_trace)} traces | extract cache: {len(_extract_cache)} windows")
per_trace.head()

In [ ]:
# ---------- aggregate: oracle vs actual, paired delta, IPW-weighted population estimate ----------
def wmean(x, w):
    x = np.asarray(x, float); w = np.asarray(w, float)
    m = ~np.isnan(x)
    return (w[m] * x[m]).sum() / w[m].sum() if m.any() and w[m].sum() > 0 else float("nan")

print(f"{'cell':26s} {'n':>3} {'actual':>7} {'oracle':>7} {'delta':>7} {'p_pair':>9}  "
      f"{'act_ipw':>7} {'ora_ipw':>7}")
summary = []
for dataset_key in ("eedi", "sciq"):
    for regime in ("reasoning", "cot"):
        for model in ("deepseek", "glm"):
            sub = per_trace[(per_trace.dataset == dataset_key) & (per_trace.regime == regime)
                            & (per_trace.model == model)]
            if not len(sub):
                continue
            a, o = sub.actual_pm.values, sub.oracle_pm.values
            delta = np.nanmean(o) - np.nanmean(a)
            t, p = st.ttest_rel(o, a)
            ai, oi = wmean(a, sub.weight.values), wmean(o, sub.weight.values)
            label = f"{dataset_key}/{regime}/{model}"
            print(f"{label:26s} {len(sub):3d} {np.nanmean(a):7.3f} {np.nanmean(o):7.3f} "
                  f"{delta:7.3f} {p:9.1e}  {ai:7.3f} {oi:7.3f}")
            summary.append(dict(dataset=dataset_key, regime=regime, model=model, n=len(sub),
                                actual=np.nanmean(a), oracle=np.nanmean(o), delta=delta, p=p,
                                actual_ipw=ai, oracle_ipw=oi))
summary = pd.DataFrame(summary)
summary

In [ ]:
# ---------- headline framing + LaTeX-ready stub (DeepSeek reasoner, the primarily-reported model) ----------
# Report the IPW-weighted (population) estimates: the annotated subset stratifies on match quality,
# so the RAW subset means over-weight high-match traces (badly on SciQ, ~neutral on Eedi). The
# IPW estimate reconciles the actual pm with the paper's headline proportional_match (021).
def _headline(dataset_key, regime="reasoning", model="deepseek"):
    row = summary[(summary.dataset == dataset_key) & (summary.regime == regime)
                  & (summary.model == model)].iloc[0]
    act, ora = row.actual_ipw, row.oracle_ipw
    delta = ora - act
    remaining = 1 - act   # share of the remaining (unrecovered) human distractors proposed-but-dropped
    recov = delta / remaining if remaining > 0 else float("nan")
    print(f"[{dataset_key}/{regime}/{model}] actual pm={act:.3f} -> oracle pm={ora:.3f} "
          f"(+{delta:.3f}); an oracle ranker over proposed candidates recovers "
          f"{recov*100:.0f}% of the human distractors the model still missed.")

for dk in ("eedi", "sciq"):
    _headline(dk)

print()
print(r"% oracle proportional match (DeepSeek reasoner, IPW population estimate) -- proposed-but-not-selected")
for dk in ("eedi", "sciq"):
    row = summary[(summary.dataset == dk) & (summary.regime == "reasoning")
                  & (summary.model == "deepseek")].iloc[0]
    print(f"% {dk.capitalize():5s}: actual {row.actual_ipw:.2f} -> oracle {row.oracle_ipw:.2f} "
          f"(paired p={row.p:.1e}, n={row.n})")

In [ ]:
# ---------- sanity assertions: oracle must be a superset of the selected set ----------
viol = per_trace[per_trace.oracle_pm + 1e-9 < per_trace.actual_pm]
print("oracle < actual violations:", len(viol))
assert per_trace.oracle_pm.dropna().between(0, 1).all(), "oracle_pm out of [0,1]"
if len(viol):
    display(viol.head())

In [ ]:
# ---------- audit dump: proposed-but-dropped matches (selected miss, oracle hit) ----------
audit = []
for _, r in per_trace[per_trace.oracle_pm > per_trace.actual_pm].head(12).iterrows():
    ds = datasets_by_datafolder[DATAFOLDER[r.dataset]]
    dpid = int(r.Id)
    problem = ds[dpid]["Problem"]["Question"]
    correct = ds[dpid]["Choices"]["CorrectAnswer"]
    gts = gt_distractors(ds, dpid)
    csv_path = f"{DATAFOLDER[r.dataset]}/joint_results/annotated/{r.run}_{r.bucket}_annot_parsed.csv"
    row = pd.read_csv(csv_path)
    row = row[row.Id == dpid].iloc[0]
    windows = inst_windows(row.get("annotation", ""))
    cand = extract_candidates(problem, correct, windows)
    selected = selected_distractors(load_responses(DATAFOLDER[r.dataset], r.run), dpid)
    checker = checker_by_dataset[r.dataset]
    for g in gts:
        hit = [c for c in cand if checker.is_equal(problem, c, g)]
        is_selected = any(checker.is_equal(problem, s, g) for s in selected)
        if hit and not is_selected:   # genuinely proposed in reasoning but dropped from final set
            audit.append({"dataset": r.dataset, "model": r.model, "regime": r.regime, "Id": dpid,
                          "gt_distractor": g, "matched_candidate": hit[0]})
print(len(audit), "proposed-but-dropped GT matches (sample of <=12 traces)")
pd.DataFrame(audit)

In [ ]:
# ---------- cross-check vs manual discards_valid (063), DeepSeek reasoner, low-match bucket ----------
def parse_dropped(path):
    if not os.path.exists(path):
        return []
    txt = open(path).read()
    return [int(m.group(1)) for m in re.finditer(r"^DROPPED:\s*(\d+)", txt, re.M)]

for dataset_key in ("eedi", "sciq"):
    manual_path = f"manual_inspection/discards_valid/{DATAFOLDER[dataset_key]}/deepseek_reasoner_naive_reasoner_joint.txt"
    manual = parse_dropped(manual_path)
    sub = per_trace[(per_trace.dataset == dataset_key) & (per_trace.model == "deepseek")
                    & (per_trace.regime == "reasoning") & (per_trace.bucket == "low_match_solvable")]
    # per-problem GT recovered by the oracle but not selected = (oracle_pm - actual_pm) * n_gt
    dropped_auto = ((sub.oracle_pm - sub.actual_pm) * sub.n_gt).round()
    m_avg = np.mean(manual) if manual else float("nan")
    print(f"{dataset_key}: manual DROPPED avg={m_avg:.2f} (n={len(manual)})  |  "
          f"auto dropped-but-proposed avg={dropped_auto.mean():.2f} (n={len(sub)})")

# How often are valid distractors dropped by `PLAUS` / `DISCR`?

Attributes part of the selection loss from Part 1 to the model's own quality-control checks (Appendix
C.5, Table 28). For every `PLAUS`/`DISCR` tag we run one LLM call (full `gpt-4.1`) over a window ending
at the tag, returning the candidate distractor value it assesses (or `NONE` for set-level remarks) and
its sentiment toward keeping that candidate (`+`/`-`/`0`). A candidate is **valid** when it matches a
human-authored distractor; *dropped* = a valid candidate judged NEGATIVE. Validated on 100 held-out
checks (`manual_inspection/plaus_discr_validation/sentiment_validation.csv`, ~93% agreement).

In [ ]:
# ---------- PLAUS/DISCR sentiment classifier (full gpt-4.1; validated prompt v9b) ----------
from src.model_configurations import gpt_4_1_det
sent_cfg = gpt_4_1_det
sent_client = OpenAI(base_url=sent_cfg["base_url"], api_key=os.environ.get(sent_cfg["api_key_var"]))

# Window ENDS at the tag (the evaluative clause is to its left); the candidate value may be on either
# side, so we keep context on both sides (left-heavy). Other inline tags stripped; tag -> <<<CHECK>>>.
LEFT_CTX, RIGHT_CTX = 300, 160
def check_window(ann, tag_start, tag_len):
    left = _TAGRE.sub("", ann[max(0, tag_start - LEFT_CTX):tag_start]).strip()
    right = _TAGRE.sub("", ann[tag_start + tag_len:tag_start + tag_len + RIGHT_CTX]).strip()
    return (left + " <<<CHECK>>>" + ((" " + right) if right else "")).strip()

AXIS_FULL = {"PLAUS": "plausibility", "DISCR": "discriminability"}
SENT_RULE = {
 "PLAUS": (
  "STEP 2 -- SENTIMENT: what is this plausibility check's SENTIMENT toward keeping the distractor?\n"
  "A good distractor must be TEMPTING -- a wrong answer a student might actually pick.\n"
  "- POSITIVE if the check endorses it: plausible/tempting/believable/a strong distractor, OR names any real reason a "
  "student could be tempted (a shared property, a common confusion, 'sounds like', 'related to', 'contains X') -- even "
  "if the same sentence also notes it is ultimately wrong. A distractor being INCORRECT is expected and is NOT negative.\n"
  "- NEGATIVE if the check argues against using it: not tempting, too obvious, too far-fetched/obscure, 'unlikely', "
  "'not plausible', 'a student who computes would see it's wrong', 'too easy to rule out', 'less common'/'rare', OR the "
  "model compares candidates and leans AGAINST this one (favours another over it).\n"
  "- NEUTRAL if purely descriptive or undecided ('but maybe...').\n"),
 "DISCR": (
  "STEP 2 -- SENTIMENT: what is this discriminability check's SENTIMENT toward keeping the distractor?\n"
  "A good distractor must be clearly WRONG yet still worth offering (not secretly correct, not so obvious no one picks it).\n"
  "- POSITIVE if the check endorses it as a good distractor: clearly/unambiguously WRONG yet usable. Being wrong is GOOD "
  "for a distractor, so 'plausible but clearly wrong upon careful checking', 'not the first/main/primary X', 'the reverse "
  "process', 'not directly Y', or simply the wrong answer => POSITIVE.\n"
  "- NEGATIVE if the check argues against using it: it might actually be CORRECT or is indistinguishable from / equal to "
  "the correct answer ('that's correct, so not a distractor'), too close/ambiguous, an invalid answer form, so obviously "
  "wrong no one would pick it, OR the model rejects/sets it aside ('not exactly', 'doesn't fit', 'less common', 'let's avoid').\n"
  "- NEUTRAL if purely descriptive or undecided ('but maybe the question expects...').\n"),
}
def SENT_SYS(axis):
    return (f"You are analyzing an excerpt from an LLM inventing wrong answer options (distractors) for a "
            f"multiple-choice question. The marker <<<CHECK>>> is a {AXIS_FULL[axis]} check the model applied. "
            f"The check's evaluative clause ENDS at the marker, so read the text immediately to the LEFT of "
            f"<<<CHECK>>> to understand what the check concludes. Base BOTH the value and the verdict on that "
            f"left clause; text AFTER the marker is the next reasoning step and must not override it.\n"
            "STEP 1 -- VALUE: identify the single candidate distractor value that this check is assessing. Use the "
            "left clause to decide WHICH candidate is meant; the candidate's VALUE may appear either just before "
            "(left of) or just after (right of) the marker -- extract it from whichever side it appears. Output it "
            "verbatim, cleaned of labels/markup/quotes. Numbers or expressions that are part of the QUESTION itself "
            "(given terms, the sequence in the prompt, the correct answer being noted as correct) are NOT the assessed "
            "distractor value.\n"
            "OUTPUT NONE (do not pick any single value) when the check does not target exactly ONE candidate, i.e. any of:\n"
            "  * it evaluates the distractor SET or SEVERAL candidates together -- tell-tale words 'these', 'they', "
            "'all', 'both', 'the three': e.g. 'these are plausible', 'these are reasonable errors', 'these are all "
            "incorrect', 'these are distinct', 'all close enough to the correct answer', 'these look good'. This applies "
            "even when specific values appear earlier in the excerpt: if the clause ENDING AT the marker is a collective "
            "judgement -- including a list of several values followed by one verdict ('630, 600 and 700 are plausible', "
            "'Incorrect: 48, 190, 112. These are plausible mistakes') -- output NONE, do NOT pick one of them;\n"
            "  * it is a general/procedural/meta remark ('I need three distinct distractors', 'let me finalize', 'so "
            "any method that does not yield X is incorrect');\n"
            "  * it verifies or restates the CORRECT answer ('X is correct', 'the logic holds', 'Tom is correct').\n"
            "Only output a value when the check clearly assesses exactly ONE specific candidate distractor.\n"
            "ALWAYS give a SENTIMENT in STEP 2, even when VALUE is NONE: judge the check's stance toward the "
            "distractor(s) it discusses. A collective endorsement ('these are plausible', 'these are good distractors', "
            "'plausible but clearly wrong') is POSITIVE; a collective rejection ('these are too obvious') is NEGATIVE.\n"
            + SENT_RULE[axis]
            + "Output exactly two lines:\nVALUE: <value or NONE>\nSENTIMENT: <POSITIVE|NEGATIVE|NEUTRAL>")

_SENT_MAP = {"POSITIVE": "+", "NEGATIVE": "-", "NEUTRAL": "0"}
SENT_PROMPT_VER = "v9b"
SENT_CACHE_PATH = "cache/plaus_discr_verdicts_v2.pkl"
_sent_cache = pickle.load(open(SENT_CACHE_PATH, "rb")) if os.path.exists(SENT_CACHE_PATH) else {}

def classify_check(problem, correct, window, axis):
    """-> (value_or_None, sentiment in {'+','-','0'}). Cached to disk keyed by (prompt_ver, axis, window)."""
    key = (SENT_PROMPT_VER, axis, window)
    if key in _sent_cache:
        return _sent_cache[key]
    user = f"<question>{problem}</question>\n<correct_answer>{correct}</correct_answer>\n<excerpt>{window}</excerpt>"
    try:
        out = prompt_openai(sent_client, SENT_SYS(axis), user, sent_cfg)
    except Exception as e:
        print("classify_check error:", e)
        return (None, "0")
    val, sent = None, "0"
    for line in out.splitlines():
        s = line.strip()
        if s.upper().startswith("VALUE:"):
            v = s[6:].strip()
            val = None if v.upper() in ("NONE", "") else v
        elif s.upper().startswith("SENTIMENT:"):
            sent = _SENT_MAP.get(s[10:].strip().upper(), "0")
    _sent_cache[key] = (val, sent)
    return (val, sent)

print("sentiment cache:", len(_sent_cache), "entries")

In [ ]:
# ---------- sweep: classify every PLAUS/DISCR tag, attach humanannot (valid) + selected ----------
# 1) enumerate all PLAUS/DISCR tags with metadata + per-(run,bucket) IPW weight (as in Part 1).
# A handful of traces appear in BOTH bucket files; keep the first occurrence per (run, Id) so each
# trace's tags are counted once (mirrors Part 1's drop_duplicates(subset=["run","Id"])).
sent_tasks = []
_seen_traces = set()
for dataset_key, regime, model in CELLS:
    data_folder = DATAFOLDER[dataset_key]; ds = datasets_by_datafolder[data_folder]
    run = STEMS[(model, regime)]
    n_high, n_low = population_bucket_sizes(data_folder, run)
    for bucket, n_pop in (("high_match_solvable", n_high), ("low_match_solvable", n_low)):
        path = f"{data_folder}/joint_results/annotated/{run}_{bucket}_annot_parsed.csv"
        assert "long_problems" not in path
        if not os.path.exists(path):
            print("missing:", path); continue
        df = pd.read_csv(path); w = n_pop / len(df) if len(df) else 0.0
        for _, r in df.iterrows():
            ann = r.get("annotation", "")
            if not isinstance(ann, str): continue
            dpid = int(r["Id"])
            if (run, dpid) in _seen_traces: continue
            _seen_traces.add((run, dpid))
            problem = ds[dpid]["Problem"]["Question"]; correct = ds[dpid]["Choices"]["CorrectAnswer"]
            for mt in re.finditer(r"<PLAUS>|<DISCR>", ann):
                sent_tasks.append(dict(dataset=dataset_key, regime=regime, model=model, run=run, bucket=bucket,
                                       Id=dpid, weight=w, axis=mt.group(0)[1:-1], tag_pos=mt.start(),
                                       problem=problem, correct=correct,
                                       window=check_window(ann, mt.start(), len(mt.group(0)))))
print(f"{len(sent_tasks)} PLAUS/DISCR tags")

# 2) warm the sentiment cache over UNIQUE (axis, window) in parallel (full gpt-4.1)
uniq = list({(t["axis"], t["window"]): t for t in sent_tasks}.values())
todo = [t for t in uniq if (SENT_PROMPT_VER, t["axis"], t["window"]) not in _sent_cache]
print(f"classify: {len(uniq)} unique windows, {len(todo)} new calls")
if todo:
    with ThreadPoolExecutor(max_workers=48) as ex:
        list(ex.map(lambda t: classify_check(t["problem"], t["correct"], t["window"], t["axis"]), todo))
    pickle.dump(_sent_cache, open(SENT_CACHE_PATH, "wb"))

# 3) warm the equality-checker cache for humanannot + selected over UNIQUE pairs (gpt-4.1-mini)
def _gts(dk, dpid): return gt_distractors(datasets_by_datafolder[DATAFOLDER[dk]], dpid)
pairs = {}
for t in sent_tasks:
    val, _ = classify_check(t["problem"], t["correct"], t["window"], t["axis"])
    if not val: continue
    responses = load_responses(DATAFOLDER[t["dataset"]], t["run"])
    others = list(_gts(t["dataset"], t["Id"])) + selected_distractors(responses, t["Id"])
    for other in others:
        pairs[(t["dataset"], t["problem"], val, other)] = checker_by_dataset[t["dataset"]]
print(f"equality pairs to ensure: {len(pairs)}")
with ThreadPoolExecutor(max_workers=48) as ex:
    list(ex.map(lambda kv: kv[1].is_equal(kv[0][1], kv[0][2], kv[0][3]), pairs.items()))
math_checker.save(MATH_CHECK_PATH); sci_checker.save(SCI_CHECK_PATH)

# 4) assemble per-candidate frame (everything cached now)
rows = []
for t in sent_tasks:
    val, sent = classify_check(t["problem"], t["correct"], t["window"], t["axis"])
    ck = checker_by_dataset[t["dataset"]]
    responses = load_responses(DATAFOLDER[t["dataset"]], t["run"])
    gts = _gts(t["dataset"], t["Id"]); sel = selected_distractors(responses, t["Id"])
    humanannot = bool(val) and any(ck.is_equal(t["problem"], val, g) for g in gts)
    selected = bool(val) and any(ck.is_equal(t["problem"], val, s) for s in sel)
    rows.append(dict(dataset=t["dataset"], regime=t["regime"], model=t["model"], run=t["run"], bucket=t["bucket"],
                     Id=t["Id"], weight=t["weight"], axis=t["axis"], tag_pos=t["tag_pos"],
                     value=val, sentiment=sent, humanannot=humanannot, selected=selected))
per_check = pd.DataFrame(rows)
per_check.to_csv("cache/plaus_discr_per_candidate.csv", index=False)
print(f"\n{len(per_check)} checks | value!=NONE {int(per_check.value.notna().sum())} | "
      f"sentiment {per_check.sentiment.value_counts().to_dict()} | "
      f"valid {int(per_check.humanannot.sum())}")
per_check.head()

In [ ]:
# ---------- aggregate: how often are VALID distractors judged NEGATIVE by PLAUS / DISCR ----------
# Unit = a PLAUS/DISCR check targeting a specific candidate (value != NONE). VALID = value matches a
# ground-truth distractor. "Dropped" = NEGATIVE sentiment. IPW-weighted by per-(run,bucket) sampling
# weight (Appendix C.4/C.5). Contrast with INVALID candidates: a working check should be NEGATIVE far
# more often on invalid candidates than on valid (human-authored) ones.
def wrate(mask, weights):
    m = np.asarray(mask, float); w = np.asarray(weights, float)
    return (w * m).sum() / w.sum() if w.sum() > 0 else float("nan")

pc = per_check[per_check.value.notna()].copy()
pc["neg"] = pc.sentiment == "-"

print(f"{'cell':24s} {'axis':5s} {'nV':>4} {'valid_drop%':>11} {'nI':>4} {'inval_drop%':>11}")
rep = []
for dataset_key in ("eedi", "sciq"):
    for regime in ("reasoning", "cot"):
        for model in ("deepseek", "glm"):
            base = pc[(pc.dataset == dataset_key) & (pc.regime == regime) & (pc.model == model)]
            if not len(base): continue
            for axis in ("PLAUS", "DISCR", "BOTH"):
                s = base if axis == "BOTH" else base[base.axis == axis]
                v, iv = s[s.humanannot], s[~s.humanannot]
                vd, ivd = wrate(v.neg, v.weight), wrate(iv.neg, iv.weight)
                if axis == "BOTH":
                    print(f"{dataset_key+'/'+regime+'/'+model:24s} {axis:5s} {len(v):4d} {vd*100:10.0f}% "
                          f"{len(iv):4d} {ivd*100:10.0f}%")
                rep.append(dict(dataset=dataset_key, regime=regime, model=model, axis=axis,
                                n_valid=len(v), valid_drop=vd, n_invalid=len(iv), invalid_drop=ivd))
report = pd.DataFrame(rep)
report[report.axis != "BOTH"]

In [ ]:
# ---------- headline + selection-conditioned cut (DeepSeek reasoner, the primarily-reported model) ----------
# Of the VALID distractors the model proposed AND evaluated with a check, what share did each check
# judge NEGATIVE (= would drop)? And of those, how many did NOT survive into the final selected set
# (the check's negative verdict is consistent with the observed drop)?
for dataset_key in ("eedi", "sciq"):
    base = pc[(pc.dataset == dataset_key) & (pc.regime == "reasoning") & (pc.model == "deepseek") & pc.humanannot]
    print(f"\n[{dataset_key}] valid distractors evaluated by a check: n={len(base)}")
    for axis in ("PLAUS", "DISCR", "BOTH"):
        s = base if axis == "BOTH" else base[base.axis == axis]
        if not len(s): continue
        drop = wrate(s.neg, s.weight)
        neg_unsel = wrate(s.neg & ~s.selected, s.weight)
        print(f"   {axis:5s}: n={len(s):3d}  NEGATIVE(drop) {drop*100:4.0f}%  "
              f"| NEGATIVE & not in final set {neg_unsel*100:4.0f}%")

# invalid-vs-valid sanity (pooled DeepSeek reasoner): checks should reject invalid candidates more
for dataset_key in ("eedi", "sciq"):
    base = pc[(pc.dataset == dataset_key) & (pc.regime == "reasoning") & (pc.model == "deepseek")]
    v, iv = base[base.humanannot], base[~base.humanannot]
    print(f"[{dataset_key}] NEGATIVE rate — valid {wrate(v.neg, v.weight)*100:.0f}%  vs  "
          f"invalid {wrate(iv.neg, iv.weight)*100:.0f}%  (invalid should be higher)")

print()
print(r"% valid distractors judged NEGATIVE (dropped) by PLAUS / DISCR — DeepSeek reasoner, IPW population estimate")
for dataset_key in ("eedi", "sciq"):
    for axis in ("PLAUS", "DISCR"):
        row = report[(report.dataset == dataset_key) & (report.regime == "reasoning")
                     & (report.model == "deepseek") & (report.axis == axis)].iloc[0]
        print(f"% {dataset_key.capitalize():5s} {axis}: {row.valid_drop*100:.0f}% of {row.n_valid} valid "
              f"(vs {row.invalid_drop*100:.0f}% of {row.n_invalid} invalid)")

In [ ]:
# ---------- audit: valid distractors the model proposed then judged NEGATIVE (dropped by its own check) ----------
aud = per_check[(per_check.humanannot) & (per_check.sentiment == "-")].copy()
print(f"{len(aud)} valid distractors received a NEGATIVE check "
      f"({int((~aud.selected).sum())} of them also absent from the final selected set)")
display(aud[["dataset", "model", "regime", "Id", "axis", "value", "selected"]].head(15))

# assertions
assert per_check.sentiment.isin(["+", "-", "0"]).all(), "bad sentiment token"
assert not per_check.duplicated(subset=["run", "Id", "tag_pos", "axis"]).any(), "duplicate check rows"
assert per_check.loc[per_check.humanannot, "value"].notna().all(), "valid row without a value"
print("assertions passed | per-candidate rows:", len(per_check))